# 📊 ID-VLM — Notebook 04: Evaluation & Failure Analysis

**Goal:** Evaluate the fine-tuned model, compare against baseline, and document failure modes.

This notebook produces the final results tables and charts for the README and resume bullet.

---

## What this notebook does:
1. Load the fine-tuned model from Notebook 03
2. Run inference on the full test set
3. Compare against baseline (Notebook 02)
4. Generate breakdown charts (by doc type, capture mode, field)
5. Document failure modes with qualitative examples
6. Generate final README-ready results

## 1. Setup

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install qwen-vl-utils
!pip install python-Levenshtein

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_DIR = '/content/drive/MyDrive/id-vlm'
REPO_DIR = '/content/id-vlm'

if not os.path.exists(f'{REPO_DIR}/config.py'):
    print('Cloning ID-VLM repository...')
    !git clone https://github.com/OmTilwar/ID-VLM.git {REPO_DIR}
else:
    print('Pulling latest repository updates...')
    !git -C {REPO_DIR} pull origin master

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Ensure output directories exist
os.makedirs(f'{PROJECT_DIR}/outputs/analysis', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs/finetuned', exist_ok=True)

import config
from src.dataset import load_dataset
from src.evaluate import (
    evaluate_predictions, generate_comparison_report,
    format_report_table, save_report, parse_vlm_json_output
)

# Verify prerequisites
assert os.path.exists(f'{PROJECT_DIR}/outputs/baseline/predictions.json'), \
    'Baseline predictions not found! Run Notebook 02 first.'

print(f'GPU: {torch.cuda.get_device_name(0)}')
print('✅ Setup complete')

## 2. Load Fine-Tuned Model

In [ ]:
from unsloth import FastVisionModel

# Try loading merged model first, fall back to base + LoRA
merged_dir = f'{PROJECT_DIR}/outputs/checkpoints/merged_model'
lora_dir = f'{PROJECT_DIR}/outputs/checkpoints/lora_adapter'

if os.path.exists(merged_dir):
    print(f'Loading merged model from {merged_dir}...')
    model, tokenizer = FastVisionModel.from_pretrained(
        merged_dir,
        load_in_4bit=True,
    )
    print('✅ Merged model loaded')
elif os.path.exists(lora_dir):
    print(f'Loading base model + LoRA adapter from {lora_dir}...')
    model, tokenizer = FastVisionModel.from_pretrained(
        'unsloth/Qwen2-VL-2B-Instruct-bnb-4bit',
        load_in_4bit=True,
    )
    # Load LoRA adapter
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, lora_dir)
    print('✅ Base model + LoRA adapter loaded')
else:
    raise FileNotFoundError('No fine-tuned model found! Run Notebook 03 first.')

FastVisionModel.for_inference(model)

## 3. Run Fine-Tuned Inference on Test Set

In [ ]:
from qwen_vl_utils import process_vision_info

def run_inference(model, tokenizer, image_path, prompt):
    """Run inference on a single image."""
    image = Image.open(image_path).convert('RGB')
    
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': prompt},
        ],
    }]
    
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = tokenizer(text=[input_text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to(model.device)
    
    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=False)
    elapsed = time.time() - start
    
    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
    output_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return output_text.strip(), elapsed

print('Inference function ready.')

In [ ]:
# Load test data
test_data = load_dataset(f'{PROJECT_DIR}/data/processed/test.jsonl')
print(f'Test samples: {len(test_data)}')

# Run inference
ft_predictions = []
total_time = 0

print(f'\nRunning fine-tuned model on {len(test_data)} test samples...')

for i, sample in enumerate(test_data):
    metadata = sample.get('metadata', {})
    image_path = metadata.get('image_path', '')
    
    if not os.path.exists(image_path):
        ft_predictions.append({
            'raw_output': '{"error": "image not found"}',
            'metadata': metadata, 'time_s': 0.0, 'sample_index': i,
        })
        continue
    
    try:
        output_text, elapsed = run_inference(model, tokenizer, image_path, config.EXTRACTION_PROMPT)
        total_time += elapsed
        
        ft_predictions.append({
            'raw_output': output_text,
            'metadata': metadata, 'time_s': elapsed, 'sample_index': i,
        })
        
        if (i + 1) % 5 == 0 or i == 0:
            print(f'  [{i+1}/{len(test_data)}] {metadata.get("doc_type", "?")} ({elapsed:.1f}s)')
    except Exception as e:
        print(f'  ❌ Error on sample {i}: {e}')
        ft_predictions.append({
            'raw_output': f'{{"error": "{str(e)}"}}',
            'metadata': metadata, 'time_s': 0.0, 'sample_index': i,
        })

print(f'\n✅ Done! Average inference: {total_time/len(test_data):.1f}s/sample')

## 4. Evaluate & Compare Against Baseline

In [ ]:
# Build ground truth
ground_truths = []
for sample in test_data:
    gt_text = sample['messages'][1]['content'][0]['text']
    try:
        fields = json.loads(gt_text)
    except json.JSONDecodeError:
        fields = {}
    ground_truths.append({'fields': fields, 'metadata': sample.get('metadata', {})})

# Evaluate fine-tuned model
ft_report = evaluate_predictions(ft_predictions, ground_truths)

print('FINE-TUNED MODEL RESULTS')
print(format_report_table(ft_report))

In [ ]:
# Load baseline results
with open(f'{PROJECT_DIR}/outputs/baseline/predictions.json', 'r') as file_obj:
    baseline_predictions = json.load(file_obj)

# Align counts
n = min(len(baseline_predictions), len(ft_predictions), len(ground_truths))
baseline_predictions = baseline_predictions[:n]
ft_predictions_aligned = ft_predictions[:n]
ground_truths_aligned = ground_truths[:n]

# Evaluate baseline
baseline_report = evaluate_predictions(baseline_predictions, ground_truths_aligned)

# Generate comparison
comparison = generate_comparison_report(baseline_report, ft_report)

# Display summary
print('=' * 70)
print('BASELINE vs. FINE-TUNED COMPARISON')
print('=' * 70)
print(f'{"Metric":<25} {"Baseline":>12} {"Fine-Tuned":>12} {"Delta":>12}')
print('-' * 70)

b_metrics = baseline_report['overall']
ft_metrics = ft_report['overall']
delta_metrics = comparison['overall']['delta']

print(f'{"Field Exact Match":<25} {b_metrics["mean_exact_match"]:>11.1%} {ft_metrics["mean_exact_match"]:>11.1%} {delta_metrics["mean_exact_match"]:>+11.1%}')
print(f'{"Mean CER":<25} {b_metrics["mean_cer"]:>11.4f} {ft_metrics["mean_cer"]:>11.4f} {delta_metrics["mean_cer"]:>+11.4f}')
print(f'{"Document Accuracy":<25} {b_metrics["document_accuracy"]:>11.1%} {ft_metrics["document_accuracy"]:>11.1%} {delta_metrics["document_accuracy"]:>+11.1%}')
print(f'{"JSON Parse Rate":<25} {baseline_report["json_parse_rate"]:>11.1%} {ft_report["json_parse_rate"]:>11.1%} {comparison["json_parse_rate"]["delta"]:>+11.1%}')
print('=' * 70)

## 5. Detailed Breakdown Charts

In [ ]:
# ── Side-by-side comparison charts ──
os.makedirs(f'{PROJECT_DIR}/outputs/analysis', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Overall metrics comparison
ax = axes[0, 0]
metrics = ['Field Exact Match', 'Document Accuracy', 'JSON Parse Rate']
baseline_vals = [b_metrics['mean_exact_match'], b_metrics['document_accuracy'], baseline_report['json_parse_rate']]
finetuned_vals = [ft_metrics['mean_exact_match'], ft_metrics['document_accuracy'], ft_report['json_parse_rate']]

x = np.arange(len(metrics))
width = 0.35
bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline', color='#EF5350', alpha=0.8)
bars2 = ax.bar(x + width/2, finetuned_vals, width, label='Fine-Tuned', color='#42A5F5', alpha=0.8)
ax.set_ylabel('Rate')
ax.set_title('Overall Performance')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=9)
ax.set_ylim(0, 1.1)
ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{bar.get_height():.1%}', ha='center', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{bar.get_height():.1%}', ha='center', fontsize=8)

# 2. By document type
ax = axes[0, 1]
doc_types = sorted(set(list(baseline_report['by_doc_type'].keys()) + list(ft_report['by_doc_type'].keys())))
b_by_dt = [baseline_report['by_doc_type'].get(dt, {}).get('mean_exact_match', 0) for dt in doc_types]
f_by_dt = [ft_report['by_doc_type'].get(dt, {}).get('mean_exact_match', 0) for dt in doc_types]

x = np.arange(len(doc_types))
ax.bar(x - width/2, b_by_dt, width, label='Baseline', color='#EF5350', alpha=0.8)
ax.bar(x + width/2, f_by_dt, width, label='Fine-Tuned', color='#42A5F5', alpha=0.8)
ax.set_ylabel('Field Exact Match')
ax.set_title('By Document Type')
ax.set_xticks(x)
ax.set_xticklabels(doc_types, fontsize=8, rotation=15)
ax.set_ylim(0, 1.1)
ax.legend()

# 3. By capture mode
ax = axes[1, 0]
modes = sorted(set(list(baseline_report['by_capture_mode'].keys()) + list(ft_report['by_capture_mode'].keys())))
b_by_cm = [baseline_report['by_capture_mode'].get(cm, {}).get('mean_exact_match', 0) for cm in modes]
f_by_cm = [ft_report['by_capture_mode'].get(cm, {}).get('mean_exact_match', 0) for cm in modes]

x = np.arange(len(modes))
ax.bar(x - width/2, b_by_cm, width, label='Baseline', color='#EF5350', alpha=0.8)
ax.bar(x + width/2, f_by_cm, width, label='Fine-Tuned', color='#42A5F5', alpha=0.8)
ax.set_ylabel('Field Exact Match')
ax.set_title('By Capture Mode')
ax.set_xticks(x)
ax.set_xticklabels(modes)
ax.set_ylim(0, 1.1)
ax.legend()

# 4. By field name (CER comparison)
ax = axes[1, 1]
all_fields = sorted(set(list(baseline_report['by_field'].keys()) + list(ft_report['by_field'].keys())))
b_cer = [baseline_report['by_field'].get(fn, {}).get('mean_cer', 1.0) for fn in all_fields]
f_cer = [ft_report['by_field'].get(fn, {}).get('mean_cer', 1.0) for fn in all_fields]

x = np.arange(len(all_fields))
ax.bar(x - width/2, b_cer, width, label='Baseline', color='#EF5350', alpha=0.8)
ax.bar(x + width/2, f_cer, width, label='Fine-Tuned', color='#42A5F5', alpha=0.8)
ax.set_ylabel('Mean CER (lower is better)')
ax.set_title('CER by Field Name')
ax.set_xticks(x)
ax.set_xticklabels(all_fields, fontsize=7, rotation=30, ha='right')
ax.legend()

plt.suptitle('ID-VLM: Baseline vs. Fine-Tuned Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/analysis/comparison_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {PROJECT_DIR}/outputs/analysis/comparison_charts.png')

In [ ]:
# Look at specific failure examples
print('=' * 60)
print('FAILURE ANALYSIS: Worst Predictions')
print('=' * 60)

# Find samples with worst CER
sample_results = ft_report.get('sample_results', [])
worst = sorted(sample_results, key=lambda x: x['field_accuracy']['mean_cer'], reverse=True)[:5]

for i, result in enumerate(worst):
    print(f'\n--- Failure {i+1} ---')
    print(f'Doc type:     {result["doc_type"]}')
    print(f'Capture mode: {result["capture_mode"]}')
    print(f'JSON valid:   {result["json_valid"]}')
    print(f'Mean CER:     {result["field_accuracy"]["mean_cer"]:.3f}')
    print(f'Exact match:  {result["field_accuracy"]["exact_match_rate"]:.1%}')
    print(f'Raw output:   {result["raw_output"][:200]}')
    print()
    for field, data in result['field_accuracy']['per_field'].items():
        status = '✅' if data['match'] else '❌'
        print(f'  {status} {field}: "{data["predicted"]}" vs "{data["ground_truth"]}" (CER={data["cer"]:.3f})')

## 6. Failure Analysis

"Investigate failures, develop hypotheses" — directly from HyperVerge's JD.

In [ ]:
# Categorize failures
sample_results = ft_report.get('sample_results', [])

failures = {'json_parse': [], 'partial_match': [], 'total_miss': [], 'perfect': []}

for i, result in enumerate(sample_results):
    fa = result['field_accuracy']
    if not result['json_valid']:
        failures['json_parse'].append(result)
    elif fa['exact_match_rate'] == 0.0:
        failures['total_miss'].append(result)
    elif fa['all_correct']:
        failures['perfect'].append(result)
    else:
        failures['partial_match'].append(result)

total = len(sample_results)
print('FAILURE CATEGORIZATION')
print('=' * 50)
print(f'  ✅ Perfect (all fields correct):  {len(failures["perfect"]):>4} ({100*len(failures["perfect"])/total:.1f}%)')
print(f'  🟡 Partial match:                 {len(failures["partial_match"]):>4} ({100*len(failures["partial_match"])/total:.1f}%)')
print(f'  🔴 Total miss (0% match):         {len(failures["total_miss"]):>4} ({100*len(failures["total_miss"])/total:.1f}%)')
print(f'  ⚠️  JSON parse failure:            {len(failures["json_parse"]):>4} ({100*len(failures["json_parse"])/total:.1f}%)')
print(f'  ──────────────────────────────────────────────')
print(f'  Total:                            {total:>4}')

In [ ]:
# Analyze which fields are hardest
print('\nHARDEST FIELDS (by CER, fine-tuned model)')
print('=' * 50)

field_stats = ft_report.get('by_field', {})
sorted_fields = sorted(field_stats.items(), key=lambda x: x[1].get('mean_cer', 0), reverse=True)

for fname, stats in sorted_fields:
    cer = stats.get('mean_cer', 0)
    em = stats.get('exact_match_rate', 0)
    n = stats.get('n_samples', 0)
    difficulty = '🟢' if cer < 0.1 else ('🟡' if cer < 0.3 else '🔴')
    print(f'  {difficulty} {fname:<20} CER={cer:.3f}  EM={em:.1%}  (n={n})')

print()
print('Hypothesis: Date fields and document numbers are typically hardest')
print('because they contain dense character sequences with separators.')

In [ ]:
# Qualitative failure examples
print('\nQUALITATIVE FAILURE EXAMPLES')
print('=' * 70)

# Show worst partial-match examples
partial_sorted = sorted(
    failures['partial_match'],
    key=lambda x: x['field_accuracy']['mean_cer'],
    reverse=True
)[:5]

for i, result in enumerate(partial_sorted):
    print(f'\n--- Failure Example {i+1} ---')
    print(f'Doc type:     {result["doc_type"]}')
    print(f'Capture mode: {result["capture_mode"]}')
    print(f'Mean CER:     {result["field_accuracy"]["mean_cer"]:.3f}')
    print(f'Exact match:  {result["field_accuracy"]["exact_match_rate"]:.1%}')
    
    for field, data in result['field_accuracy']['per_field'].items():
        if not data['match']:
            print(f'  ❌ {field}:')
            print(f'     Predicted: "{data["predicted"]}"')
            print(f'     Expected:  "{data["ground_truth"]}"')
            print(f'     CER: {data["cer"]:.3f}')

In [ ]:
# Failure analysis by capture condition
print('\nCAPTURE CONDITION ANALYSIS')
print('=' * 50)
print('Hypothesis: Scans are easiest, video frames are hardest.')
print()

for cm in sorted(ft_report.get('by_capture_mode', {}).keys()):
    stats = ft_report['by_capture_mode'][cm]
    b_stats = baseline_report['by_capture_mode'].get(cm, {})
    
    em = stats.get('mean_exact_match', 0)
    b_em = b_stats.get('mean_exact_match', 0)
    delta = em - b_em
    
    bar = '█' * int(em * 30) + '░' * int((1-em) * 30)
    print(f'  {cm:<14} {bar} {em:.1%} (Δ {delta:+.1%})')

## 7. Save Final Results

In [ ]:
# Save everything
analysis_dir = f'{PROJECT_DIR}/outputs/analysis'
finetuned_dir = f'{PROJECT_DIR}/outputs/finetuned'
os.makedirs(analysis_dir, exist_ok=True)
os.makedirs(finetuned_dir, exist_ok=True)

# Save fine-tuned predictions
with open(f'{finetuned_dir}/predictions.json', 'w') as file_obj:
    json.dump(ft_predictions, file_obj, indent=2, ensure_ascii=False)

# Save reports
save_report(ft_report, f'{finetuned_dir}/report.json')

# Save comparison
with open(f'{analysis_dir}/comparison.json', 'w') as file_obj:
    json.dump(comparison, file_obj, indent=2, default=str)

print('✅ All results saved!')

In [ ]:
# Generate README-ready results tables

b_metrics = baseline_report['overall']
ft_metrics = ft_report['overall']
delta_metrics = comparison['overall']['delta']

print('\n' + '=' * 70)
print('README-READY RESULTS (copy-paste these into README.md)')
print('=' * 70)

print('\n### Key Results\n')
print('| Metric | Zero-Shot Baseline | Fine-Tuned (LoRA) | Δ |')
print('|---|---|---|---|')
print(f'| Field Exact Match | {b_metrics["mean_exact_match"]:.1%} | {ft_metrics["mean_exact_match"]:.1%} | {delta_metrics["mean_exact_match"]:+.1%} |')
print(f'| Character Error Rate | {b_metrics["mean_cer"]:.4f} | {ft_metrics["mean_cer"]:.4f} | {delta_metrics["mean_cer"]:+.4f} |')
print(f'| JSON Parse Rate | {baseline_report["json_parse_rate"]:.1%} | {ft_report["json_parse_rate"]:.1%} | {comparison["json_parse_rate"]["delta"]:+.1%} |')
print(f'| Document-Level Accuracy | {b_metrics["document_accuracy"]:.1%} | {ft_metrics["document_accuracy"]:.1%} | {delta_metrics["document_accuracy"]:+.1%} |')

print('\n### By Capture Condition\n')
print('| Condition | Baseline | Fine-Tuned |')
print('|---|---|---|')
for cm in sorted(ft_report.get('by_capture_mode', {}).keys()):
    b_val = baseline_report['by_capture_mode'].get(cm, {}).get('mean_exact_match', 0)
    f_val = ft_report['by_capture_mode'][cm].get('mean_exact_match', 0)
    print(f'| {cm} | {b_val:.1%} | {f_val:.1%} |')

print('\n### By Document Type\n')
print('| Document Type | Baseline | Fine-Tuned |')
print('|---|---|---|')
for dt in sorted(ft_report.get('by_doc_type', {}).keys()):
    b_val = baseline_report['by_doc_type'].get(dt, {}).get('mean_exact_match', 0)
    f_val = ft_report['by_doc_type'][dt].get('mean_exact_match', 0)
    print(f'| {dt} | {b_val:.1%} | {f_val:.1%} |')

In [ ]:
# Generate resume bullet
improvement = delta_metrics['mean_exact_match'] * 100

print('\n' + '=' * 70)
print('RESUME BULLET')
print('=' * 70)
print()
print(f'ID-VLM: VLM Fine-Tuning for Identity Document Understanding | Python, PyTorch, Qwen2-VL, Unsloth, OpenCV')
print(f'• Fine-tuned Qwen2-VL-2B via LoRA (Unsloth) on MIDV-2020 for structured field extraction from identity')
print(f'  documents, improving {improvement:.0f}% field accuracy over zero-shot baseline across scans, photos, and video frames.')
print(f'• Built synthetic tamper-detection pipeline (OpenCV splice/blur augmentation) and evaluation harness;')
print(f'  documented failure modes by capture condition and document type.')
print()
print('=' * 70)
print('\n🎉 Project complete! Update your README.md with these results.')